# 2D Kalman Filter

The purpose of this notebook is for me to map out the concept of Kalman filtering using a concrete example, interpretable time series data and code without using an explicit filtering library.


## Overview of The Filter

This class is a small Constant Velocity 2D Kalman filter for a moving object, like a point on a map or a tracked target in a plane. The basic idea is simple: at each time step, we make a prediction about where the object should be next, then we compare that prediction against a noisy measurement and adjust if needed.
 
This class keeps track of both: 

- the object's current state: position and velocity
- and how uncertain we are about that state

The state is stored as:

- `x`: horizontal position
- `y`: vertical position
- `vx`: velocity component x-axis
- `vy`: velocity component y-axis


### Line by line

- `self.dt = dt`: this is the time step between updates. If the data arrives every 0.1 seconds, then `dt = 0.1`.
- `self.x = np.zeros((4, 1))`: start with a belief that the object is at the origin and not moving.
- `self.P = np.eye(4) * 100`: the covariance matrix stores uncertainty. A large value means “we are not very sure yet.”
- `self.F = ...`: this is the motion model. It says: if the object is moving at velocity `(vx, vy)`, then after one time step it should move by `vx * dt` in x and `vy * dt` in y.
- `self.Q = np.eye(4) * 0.1`: process noise. Even with a motion model, the real world is noisy. We add a little uncertainty so the filter does not become overconfident.
- `self.R = np.eye(4) * 4`: measurement noise. Observed positions are noisy, so we assume the measurements are imperfect.
- `self.H = ...`: this says “when we measure the object, we only observe its position, not its velocity.”

### Concrete example

Imagine a point starts at `(0, 0)` and is moving right at 2 units per second and up at 1 unit per second. That means:

```python
x = [0, 0, 2, 1]
```

With `dt = 1`, the motion model says:

- new x = old x + vx * dt = 0 + 2 * 1 = 2
- new y = old y + vy * dt = 0 + 1 * 1 = 1
- velocity stays roughly the same

So the filter predicts the state should become:

```python
[2, 1, 2, 1]
```

This is exactly what the matrix `F` is doing. It propagates the previous belief forward in time.

The uncertainty update uses:

```python
self.P = self.F @ self.P @ self.F.T + self.Q
```

That means: take the old uncertainty, carry it through the motion model, then add process noise to reflect uncertainty in the model itself. This is the standard Kalman filter prediction step.


In [1]:
%pip install numpy

  Using cached numpy-2.5.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp314-cp314-macosx_14_0_arm64.whl (5.5 MB)

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import csv
import io

In [ ]:
class PlanarKalmanFilter:
    def __init__(self, dt: float):
        self.dt = dt
        self.x = np.zeros((4, 1)) # State vector [x, y, vx, vy]
        self.P = np.eye(4) * 100 # State covariance matrix (initial uncertainty)
        self.F = np.array([[1, 0, dt, 0],  # State transition matrix
                           [0, 1, 0, dt],
                           [0, 0, 1, 0],
                           [0, 0, 0, 1]])
        self.Q = np.eye(4) * 0.1 # Process noise covariance matrix
        self.R = np.eye(4) * 4 # Measurement noise covariance matrix
        self.H = np.array([ # Measurement matrix measuring position components
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

        return self.x, self.P

    def update(self, zx: float, zy: float):
        z = np.array([[zx],
                      [zy]])
        y = z - (self.H @ self.x)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P

        return self.x, self.P

    @property
    def position(self):
        return float(self.x[0]) , float(self.x[1])